# Objetivo del Script
Este script tiene como objetivo principal enriquecer un dataset de propiedades inmobiliarias (`df_venta`) con información detallada sobre la disponibilidad de servicios y condiciones de seguridad en sus respectivas colonias. Para lograr esto, se realiza una fusión de datos geográficos y estadísticos de servicios públicos (hospitales, escuelas, esparcimiento, restaurantes, transporte, parques) y datos de crimen, con el fin de proporcionar un contexto más completo para el análisis del valor de las propiedades.

# Resumen del flujo
1.  **Configuración Inicial**: Se importan las librerías necesarias para el análisis de datos, manipulación de texto y funciones de concordancia aproximada.
2.  **Carga y Extracción de Datos**: Se extrae un archivo ZIP que contiene los datasets de servicios y se cargan tanto el dataset principal de ventas de propiedades (`df_venta`) como los datasets de servicios (hospitales, escuelas, etc.) en DataFrames de Pandas.
3.  **Consolidación de Datos de Servicios**: Los DataFrames individuales de servicios se fusionan en un único DataFrame (`df_map`) que contiene la información geográfica y el conteo de servicios por colonia/asentamiento.
4.  **Normalización de Nombres de Ubicación**: Se aplica una limpieza y estandarización rigurosa a las columnas de `Colonia`, `Municipio` y `Estado` en ambos el dataset de propiedades y el dataset de servicios (`df_map`). Esto incluye la eliminación de acentos, puntuación, estandarización de abreviaturas y corrección de nombres específicos para asegurar una alta tasa de concordancia.
5.  **Fusión Inicial de Datasets**: Se realiza una fusión (`left join`) entre el dataset de propiedades (`df_precios`) y el dataset consolidado de servicios (`df_map`) utilizando `Colonia`, `Municipio` y `Estado` como claves.
6.  **Concordancia Aproximada (Fuzzy Matching)**: Para las propiedades que no encontraron una coincidencia exacta en la fusión inicial, se implementa un algoritmo de *fuzzy matching* (`rapidfuzz`) para intentar encontrar colonias similares dentro del mismo municipio y estado, aumentando así la cantidad de datos enriquecidos.
7.  **Transformación de Tipo de Datos**: La columna `Fecha` se convierte a formato `datetime` para facilitar análisis temporales.
8.  **Guardado de Resultados**: El DataFrame final enriquecido (`df_filled`) se guarda en un archivo CSV para su uso posterior.

# Archivos de Entrada
-   `/content/Servicios_redonda.zip`: Archivo ZIP que contiene los datasets de servicios.
-   `/content/Lamudi_Inmu_10.csv`: Dataset principal de propiedades inmobiliarias.
-   `/content/Servicios_redonda/Hospitales.csv`: Conteo de hospitales por área.
-   `/content/Servicios_redonda/Escuelas.csv`: Conteo de escuelas por área.
-   `/content/Servicios_redonda/Esparcimientos.csv`: Conteo de puntos de esparcimiento por área.
-   `/content/Servicios_redonda/Restaurantes.csv`: Conteo de restaurantes por área.
-   `/content/Servicios_redonda/Carpetas.csv`: Conteo de carpetas de investigación (crimen) por área.
-   `/content/Servicios_redonda/Transporte.csv`: Conteo de puntos de transporte por área.
-   `/content/Servicios_redonda/Parques.csv`: Conteo de parques por área.

# Archivos de Salida
-   `no_match_sample.csv`: Un archivo CSV que contiene una muestra de las entradas que no lograron ser coincidentes, útil para depuración.
-   `df_precios_con_servicios.csv`: El dataset final de propiedades, enriquecido con la información de servicios y seguridad.

# Librerías Utilizadas
-   `pandas` (`pd`): Para manipulación y análisis de datos, incluyendo la carga de CSVs, fusiones y limpieza de DataFrames.
-   `numpy` (`np`): Para operaciones numéricas, a menudo utilizado en conjunto con pandas.
-   `matplotlib.pyplot` (`plt`): Para la creación de visualizaciones estáticas (aunque no se usa directamente en el flujo actual, es una librería común para EDA).
-   `seaborn` (`sns`): Para la creación de visualizaciones estadísticas atractivas (similar a matplotlib, no directamente usada en el flujo actual).
-   `geopandas` (`gpd`): Importada, pero su uso para archivos `.shp` está comentado, y en su lugar se cargan CSVs.
-   `re`: Módulo para operaciones con expresiones regulares, esencial para la limpieza y normalización de texto.
-   `warnings`: Para controlar la emisión de mensajes de advertencia.
-   `unidecode`: Para convertir texto con caracteres acentuados a su equivalente sin acentos.
-   `rapidfuzz`: Para realizar concordancia de cadenas de texto aproximada (fuzzy matching), utilizada para mejorar la tasa de coincidencias entre colonias.


## 1.  **Importaciones**:

Las primeras celdas cargan las librerías necesarias. `pandas` es fundamental para el manejo de los DataFrames, `re` y `unidecode` para la limpieza de texto, y `rapidfuzz` para la concordancia aproximada.

## 2.  **Extracción de Archivos**:

Descomprime el archivo `Servicios_redonda.zip` en el directorio `/content/`, haciendo disponibles los CSVs de servicios.

## 3.  **Carga de Datos**:

Carga `Lamudi_Inmu_10.csv` en `df_venta`. Las celdas cargan los CSVs de servicios individuales (Hospitales, Escuelas, Esparcimientos, Restaurantes, Carpetas, Transporte, Parques) en sus respectivos DataFrames.

## 4.  **Consolidación de `df_map`**:

Realiza una serie de fusiones `outer` para combinar todos los DataFrames de servicios (`hospitales_shp`, `escuelas_shp`, etc.) en un único DataFrame llamado `df_map`. La fusión se basa en columnas geográficas y de identificación como `OBJECTID`, `POSTALCODE`, `ST_NAME`, `MUN_NAME`, `SETT_NAME`, `geometry`, `latitud`, y `longitud`.

## 5.  **Renombrado de Columnas**:

Renombra algunas columnas en `df_map` (ej., `Escuelas_1` a `Escuelas`, `Esparcimie` a `Esparcimiento`) para hacerlas más descriptivas y consistentes.

## 6.  **Limpieza Inicial `df_venta`**:

Eimina una fila específica de `df_venta` que parece ser un dato atípico o erróneo (`Estado` == 'IV, IX, Naucalpan de Juárez, Estado de México').

## 7.  **Preparación para Normalización**:

Las celdas crean puntos de control (`checkpoint`) de `df_precios` y `df_map` antes de la normalización intensiva, permitiendo reversiones si es necesario.

## 8.  **Normalización Detallada y Fusión**:

Bloque de código robusto para la limpieza y normalización de las columnas `Colonia`, `Municipio`, y `Estado`/`SETT_NAME`/`MUN_NAME`/`ST_NAME`. Esto incluye:
    -   Una función `clean_text` para estandarizar cadenas de texto (mayúsculas, sin acentos, sin puntos, espacios compactos).
    -   Armonización de nombres de `Estado` (ej., `CDMX` se mapea a `DISTRITO FEDERAL`).
    -   Reemplazos específicos en nombres de colonia y municipio (ej., `SECCION` por `SECC`, correcciones de `POLANCO`).
    -   Realiza un `left merge` inicial entre `df_precios` y `df_map` utilizando las columnas normalizadas de ubicación como claves.

## 9.  **Fuzzy Matching**:

Se define e invoca la función `fill_with_fuzzy` que, para las filas que no coincidieron exactamente en la fusión directa, utiliza `rapidfuzz.process.extractOne` para encontrar la colonia más similar (con una puntuación de similitud superior a `score_cutoff=92`) dentro del mismo municipio y estado en `df_map`, y luego rellena los datos de servicios correspondientes. Esto mejora significativamente la cobertura de datos.

## 10. **Diagnóstico y Guardado de No Coincidentes**:

Imprime estadísticas sobre las coincidencias y guarda un archivo `no_match_sample.csv` con las colonias que no pudieron ser coincidentes incluso después del fuzzy matching.

## 11. **Inspección del DataFrame Final**:

Muestra un resumen de la información del DataFrame `df_filled` (resultante de las fusiones y fuzzy matching), incluyendo el número de valores no nulos para cada columna de servicio, indicando la completitud de los datos.

## 12. **Conversión de `Fecha`**:

Convierte la columna `Fecha` a tipo `datetime` para permitir operaciones basadas en tiempo y análisis de series temporales.

## 13. **Configuración de Opciones de Visualización**:

Ajusta las opciones de visualización de Pandas (`pd.set_option`) para mostrar más filas y columnas al imprimir DataFrames, facilitando la inspección visual.

## 14. **Verificación de Datos Post-Normalización**:

Muestra un subconjunto de `df_filled` filtrado por el municipio 'ALVARO OBREGON' para verificar que la normalización de nombres y la fusión de datos de servicios se hayan aplicado correctamente.

## 15. **Guardado del Resultado Final**:

Finalmente, guarda el DataFrame final `df_filled`, que contiene la información de propiedades enriquecida con los servicios y seguridad, en el archivo `df_precios_con_servicios.csv`.

In [1]:
import pandas as pd # Para manipulación y análisis de datos en DataFrames
import numpy as np # Para operaciones numéricas, a menudo utilizado con pandas
import matplotlib.pyplot as plt # Para la creación de visualizaciones estáticas
import seaborn as sns # Para la creación de visualizaciones estadísticas atractivas
import geopandas as gpd # Para trabajar con datos geoespaciales (aunque su uso directo está comentado en este script)
import re # Para operaciones con expresiones regulares, útil en la limpieza de texto
import warnings # Para controlar la emisión de mensajes de advertencia
warnings.filterwarnings('ignore') # Ignorar advertencias para una salida más limpia

In [2]:
!unzip -o -q "/content/Servicios_redonda.zip" -d /content/ # Descomprime el archivo ZIP que contiene los datasets de servicios en el directorio /content/

unzip:  cannot find or open /content/Servicios_redonda.zip, /content/Servicios_redonda.zip.zip or /content/Servicios_redonda.zip.ZIP.


In [ ]:
df_venta = pd.read_csv('/content/Lamudi_Inmu_10.csv') # Carga el dataset principal de propiedades inmobiliarias

In [ ]:
# hospitales_shp = gpd.read_file('Colonias_ZMVM/Hospitales/hospitales_conteo.shp') # Línea comentada para cargar archivos .shp
hospitales_shp = pd.read_csv('/content/Servicios_redonda/Hospitales.csv') # Carga el dataset de hospitales
# Eliminar la columna de índice si existe (en este caso 'Unnamed: 0')
hospitales_shp.info()

In [ ]:
# escuelas_shp = gpd.read_file('Colonias_ZMVM/Escuelas/conteo_escuelas.shp') # Línea comentada para cargar archivos .shp
escuelas_shp = pd.read_csv('/content/Servicios_redonda/Escuelas.csv') # Carga el dataset de escuelas
escuelas_shp.info()

In [ ]:
# esparcimiento_shp = gpd.read_file('/content/Colonias_ZMVM/Esparcimiento/esparcimiento_conteo.shp') # Línea comentada para cargar archivos .shp
esparcimiento_shp = pd.read_csv('/content/Servicios_redonda/Esparcimientos.csv') # Carga el dataset de esparcimiento
esparcimiento_shp.info()

In [ ]:
# restaurantes_shp = gpd.read_file('/content/Colonias_ZMVM/Restaurantes/restaurantes_conteo.shp') # Línea comentada para cargar archivos .shp
restaurantes_shp = pd.read_csv('/content/Servicios_redonda/Restaurantes.csv') # Carga el dataset de restaurantes
restaurantes_shp.info()

In [ ]:
# crimen_shp = gpd.read_file('/content/Colonias_ZMVM/Seguridad/crimen_conteo_CDMX.shp') # Línea comentada para cargar archivos .shp
crimen_shp = pd.read_csv('/content/Servicios_redonda/Carpetas.csv') # Carga el dataset de crimen (carpetas de investigación)
crimen_shp.info()

In [ ]:
# transporte_shp = gpd.read_file('/content/Transporte/Transporte_conteo.shp') # Línea comentada para cargar archivos .shp
transporte_shp = pd.read_csv('/content/Servicios_redonda/Transporte.csv') # Carga el dataset de transporte
transporte_shp.info()

In [ ]:
parques_shp = pd.read_csv('/content/Servicios_redonda/Parques.csv') # Carga el dataset de parques
parques_shp.info()

In [ ]:
# Realiza una serie de fusiones ('outer' merge) para combinar todos los DataFrames de servicios en un único DataFrame llamado `df_map`.
# La fusión se basa en columnas geográficas y de identificación.
df_map = hospitales_shp[['OBJECTID', 'POSTALCODE', 'ST_NAME', 'MUN_NAME', 'SETT_NAME', 'Hospitales', 'geometry', 'latitud', 'longitud']].merge(
    escuelas_shp[['OBJECTID', 'POSTALCODE', 'ST_NAME', 'MUN_NAME', 'SETT_NAME', 'Escuelas_1', 'geometry', 'latitud', 'longitud']],
    on=['OBJECTID', 'POSTALCODE', 'ST_NAME', 'MUN_NAME', 'SETT_NAME', 'geometry', 'latitud', 'longitud'], how='outer')

df_map = df_map.merge(
    esparcimiento_shp[['OBJECTID', 'POSTALCODE', 'ST_NAME', 'MUN_NAME', 'SETT_NAME', 'Esparcimie', 'geometry', 'latitud', 'longitud']],
    on=['OBJECTID', 'POSTALCODE', 'ST_NAME', 'MUN_NAME', 'SETT_NAME', 'geometry', 'latitud', 'longitud'], how='outer')

df_map = df_map.merge(
    restaurantes_shp[['OBJECTID', 'POSTALCODE', 'ST_NAME', 'MUN_NAME', 'SETT_NAME', 'Restaurant', 'geometry', 'latitud', 'longitud']],
    on=['OBJECTID', 'POSTALCODE', 'ST_NAME', 'MUN_NAME', 'SETT_NAME', 'geometry', 'latitud', 'longitud'], how='outer')

df_map = df_map.merge(
    crimen_shp[['OBJECTID', 'POSTALCODE', 'ST_NAME', 'MUN_NAME', 'SETT_NAME', 'Carpetas_1', 'geometry', 'latitud', 'longitud']],
    on=['OBJECTID', 'POSTALCODE', 'ST_NAME', 'MUN_NAME', 'SETT_NAME', 'geometry', 'latitud', 'longitud'], how='outer')

df_map = df_map.merge(
    transporte_shp[['OBJECTID', 'POSTALCODE', 'ST_NAME', 'MUN_NAME', 'SETT_NAME', 'Transporte', 'geometry', 'latitud', 'longitud']],
    on=['OBJECTID', 'POSTALCODE', 'ST_NAME', 'MUN_NAME', 'SETT_NAME', 'geometry', 'latitud', 'longitud'], how='outer'
)

df_map = df_map.merge(
    parques_shp[['OBJECTID', 'POSTALCODE', 'ST_NAME', 'MUN_NAME', 'SETT_NAME', 'leisure_co', 'geometry', 'latitud', 'longitud']],
    on=['OBJECTID', 'POSTALCODE', 'ST_NAME', 'MUN_NAME', 'SETT_NAME', 'geometry', 'latitud', 'longitud'], how='outer'
)

display(df_map.head())

In [ ]:
df_map = df_map.rename(columns = { # Renombra columnas para mayor claridad y consistencia
    'Escuelas_1': 'Escuelas',
    'Esparcimie': 'Esparcimiento',
    'Restaurant': 'Restaurantes',
    'Carpetas_1': 'Carpetas',
    'leisure_co': 'Parques',
    'latitud': 'Latitud',
    'longitud': 'Longitud'
})

df_map.head()

In [ ]:
print(df_venta.shape) # Muestra la forma original del DataFrame
df_venta = df_venta[df_venta['Estado'] != 'IV, IX, Naucalpan de Juárez, Estado de México'] # Elimina una fila con datos atípicos o erróneos
print(df_venta.shape) # Muestra la forma del DataFrame después de la eliminación

In [ ]:
df_precios = df_venta.copy() # Crea una copia de df_venta para trabajar con ella

# Normaliza los nombres de las columnas de ubicación (convertir a mayúsculas y quitar espacios en blanco)
df_precios["Colonia"] = df_precios["Colonia"].str.upper().str.strip()
df_precios["Municipio"] = df_precios["Municipio"].str.upper().str.strip()
df_map["SETT_NAME"] = df_map["SETT_NAME"].str.upper().str.strip()
df_map["MUN_NAME"] = df_map["MUN_NAME"].str.upper().str.strip()

# Corregir acentos en nombres de municipios para asegurar la concordancia
df_precios['Municipio'] = df_precios['Municipio'].replace('ALVARO OBREGON', 'ÁLVARO OBREGÓN')
df_precios['Municipio'] = df_precios['Municipio'].replace('BENITO JUAREZ', 'BENITO JUÁREZ')
df_precios['Municipio'] = df_precios['Municipio'].replace('COYOACAN', 'COYOACÁN')
df_precios['Municipio'] = df_precios['Municipio'].replace('CUAUHTEMOC', 'CUAUHTÉMOC')
df_precios['Municipio'] = df_precios['Municipio'].replace('TLAHUAC', 'TLÁHUAC')
df_precios['Municipio'] = df_precios['Municipio'].replace('GUSTAVO A. MADERO', 'GUSTAVO A MADERO')


df_precios['Municipio'] = df_precios['Municipio'].replace('ALMOLOYA DE JUAREZ', 'ALMOLOYA DE JUÁREZ')
df_precios['Municipio'] = df_precios['Municipio'].replace('ALMOLOYA DEL RIO', 'ALMOLOYA DEL RÍO')
df_precios['Municipio'] = df_precios['Municipio'].replace('ATIZAPAN', 'ATIZAPÁN')
df_precios['Municipio'] = df_precios['Municipio'].replace('ATIZAPAN DE ZARAGOZA', 'ATIZAPÁN DE ZARAGOZA')
df_precios['Municipio'] = df_precios['Municipio'].replace('CHIMALHUACAN', 'CHIMALHUACÁN')
df_precios['Municipio'] = df_precios['Municipio'].replace('COACALCO DE BERRIOZABAL', 'COACALCO DE BERRIOZÁBAL')
df_precios['Municipio'] = df_precios['Municipio'].replace('COCOTITLAN', 'COCOTITLÁN')
df_precios['Municipio'] = df_precios['Municipio'].replace('CUAUTITLAN', 'CUAUTITLÁN')
df_precios['Municipio'] = df_precios['Municipio'].replace('CUAUTITLAN IZCALLI', 'CUAUTITLÁN IZCALLI')
df_precios['Municipio'] = df_precios['Municipio'].replace('JOCOTITLAN', 'JOCOTITLÁN')
df_precios['Municipio'] = df_precios['Municipio'].replace('NAUCALPAN DE JUAREZ', 'NAUCALPAN DE JUÁREZ')
df_precios['Municipio'] = df_precios['Municipio'].replace('NEZAHUALCOYOTL', 'NEZAHUALCÓYOTL')
df_precios['Municipio'] = df_precios['Municipio'].replace('NICOLAS ROMERO', 'NICOLÁS ROMERO')
df_precios['Municipio'] = df_precios['Municipio'].replace('POLOTITLAN', 'POLOTITLÁN')
df_precios['Municipio'] = df_precios['Municipio'].replace('RAYON', 'RAYÓN')
df_precios['Municipio'] = df_precios['Municipio'].replace('SAN JOSE DEL RINCON', 'SAN JOSÉ DEL RINCÓN')
df_precios['Municipio'] = df_precios['Municipio'].replace('SAN MARTIN DE LAS PIRAMIDES', 'SAN MARTÍN DE LAS PIRÁMIDES')
df_precios['Municipio'] = df_precios['Municipio'].replace('SAN SIMON DE GUERRERO', 'SAN SIMÓN DE GUERRERO')
df_precios['Municipio'] = df_precios['Municipio'].replace('SANTO TOMAS', 'SANTO TOMÁS')
df_precios['Municipio'] = df_precios['Municipio'].replace('SOYANIQUILPAN DE JUAREZ', 'SOYANIQUILPAN DE JUÁREZ')
df_precios['Municipio'] = df_precios['Municipio'].replace('TECAMAC', 'TECÁMAC')
df_precios['Municipio'] = df_precios['Municipio'].replace('TEOTIHUACAN', 'TEOTIHUACÁN')
df_precios['Municipio'] = df_precios['Municipio'].replace('TEPOTZOTLAN', 'TEPOTZOTLÁN')
df_precios['Municipio'] = df_precios['Municipio'].replace('TEXCALTITLAN', 'TEXCALTITLÁN')
df_precios['Municipio'] = df_precios['Municipio'].replace('TULTITLAN', 'TULTITLÁN')
df_precios['Municipio'] = df_precios['Municipio'].replace('VILLA DEL CARBON', 'VILLA DEL CARBÓN')
df_precios['Municipio'] = df_precios['Municipio'].replace('XONACATLAN', 'XONACATLÁN')
df_precios['Municipio'] = df_precios['Municipio'].replace('ZUMPAHUACAN', 'ZUMPAHUACÁN')

# Estandarización de nombres de colonias y asentamientos en df_map
df_map['SETT_NAME'] = df_map['SETT_NAME'].str.replace('SECCION', 'SECC')
df_map['SETT_NAME'] = df_map['SETT_NAME'].str.replace('1RA', '1A')
df_map['SETT_NAME'] = df_map['SETT_NAME'].str.replace('2DA', '2A')
df_map['SETT_NAME'] = df_map['SETT_NAME'].str.replace('3RA', '3A')
df_map['SETT_NAME'] = df_map['SETT_NAME'].str.replace('POLANCO REFORMA', 'POLANCO')
df_map['SETT_NAME'] = df_map['SETT_NAME'].str.replace('POLANCO CHAPULTEPEC', 'POLANCO')
df_map['SETT_NAME'] = df_map['SETT_NAME'].str.replace('BOSQUES', 'BOSQUE')
df_map['SETT_NAME'] = df_map['SETT_NAME'].str.replace('SELENE 1A SECC', 'SELENE')
df_map['SETT_NAME'] = df_map['SETT_NAME'].str.replace('SELENE 2A SECC', 'SELENE')
df_map['SETT_NAME'] = df_map['SETT_NAME'].str.replace('UNIDAD HAB SANTA CRUZ MEYEHUALCO', 'SANTA CRUZ MEYEHUALCO')
df_map['SETT_NAME'] = df_map['SETT_NAME'].str.replace('PUEBLO SANTA CRUZ MEYEHUALCO', 'SANTA CRUZ MEYEHUALCO')
df_map['SETT_NAME'] = df_map['SETT_NAME'].str.replace('UNIDAD HAB BELEN', 'BELEN DE LAS FLORES')
df_map['SETT_NAME'] = df_map['SETT_NAME'].str.replace('PRESIDENTES', '2A AMPL PRESIDENTES')
df_map['SETT_NAME'] = df_map['SETT_NAME'].str.replace('AMPL PRESIDENTES', '2A AMPL PRESIDENTES')
df_map['SETT_NAME'] = df_map['SETT_NAME'].str.replace('.', '', regex=False)
df_map['SETT_NAME'] = df_map['SETT_NAME'].apply(lambda x: re.sub(r'BELEN DE LAS FLORES.*', 'BELEN DE LAS FLORES', x))
df_map['SETT_NAME'] = df_map['SETT_NAME'].apply(lambda x: re.sub(r'MAGDALENA MIXHUCA.*', 'MAGDALENA MIXHUCA', x))

# Corrige el error de sintaxis en re.findall y usa indexación booleana para la asignación
condition = (df_map['SETT_NAME'].str.contains('CUAUTEPEC')) & (df_map['MUN_NAME'] == 'GUSTAVO A MADERO')
df_map.loc[condition, 'SETT_NAME'] = 'CUAUTEPEC DE MADERO'

# Estandarización de nombres de colonias en df_precios
df_precios['Colonia'] = df_precios['Colonia'].str.replace('SECCION', 'SECC')
df_precios['Colonia'] = df_precios['Colonia'].str.replace('1RA', '1A')
df_precios['Colonia'] = df_precios['Colonia'].str.replace('2DA', '2A')
df_precios['Colonia'] = df_precios['Colonia'].str.replace('3RA', '3A')
df_precios['Colonia'] = df_precios['Colonia'].str.replace('EX-', 'EX ')
df_precios['Colonia'] = df_precios['Colonia'].str.replace(' - ', ' ')
df_precios['Colonia'] = df_precios['Colonia'].str.replace('.', '', regex=False)
df_precios['Colonia'] = df_precios['Colonia'].str.replace('BOSQUES', 'BOSQUE')
df_precios['Colonia'] = df_precios['Colonia'].str.replace('CENTRAL DE ABASTO', 'CENTRAL DE ABASTOS')
df_precios['Colonia'] = df_precios['Colonia'].str.replace('UNIDAD HAB BELEN', 'BELEN DE LAS FLORES')
df_precios['Colonia'] = df_precios['Colonia'].str.replace('SAN JUAN DE ARAGON I SECC', 'SAN JUAN DE ARAGON 1A SECC')
df_precios['Colonia'] = df_precios['Colonia'].str.replace('SAN JUAN DE ARAGON II SECC', 'SAN JUAN DE ARAGON 2A SECC')
df_precios['Colonia'] = df_precios['Colonia'].str.replace('FERNANDO CASAS ALEMAN', 'CASAS ALEMAN')
df_precios['Colonia'] = df_precios['Colonia'].str.replace('CENTRO URBANO PRESIDENTE ALEMAN', 'MIGUEL ALEMAN')
df_precios['Colonia'] = df_precios['Colonia'].str.replace('SANTA FE IMSS', 'UNIDAD HAB SANTA FE IMSS')
df_precios['Colonia'] = df_precios['Colonia'].str.replace('LOMAS DE LOS ANGELES DEL PUEBLO TETELPAN     ', 'LOMAS DE LOS ANGELES TETELPAN')
df_precios['Colonia'] = df_precios['Colonia'].str.replace('LAS AGUILAS 1A SECC', 'PARQUE LAS AGUILAS')
df_precios['Colonia'] = df_precios['Colonia'].str.replace('LAS AGUILAS 2O PARQUE', 'LAS AGUILAS AMPLIACION 2O PARQUE')
df_precios['Colonia'] = df_precios['Colonia'].str.replace('LAS AGUILAS 3ER PARQUE', 'AMPL LAS AGUILAS 3ER PARQUE')
df_precios['Colonia'] = df_precios['Colonia'].str.replace('AMPLIACION SELENE', 'AMPL SELENE')
df_precios['Colonia'] = df_precios['Colonia'].str.replace('EL PIRU SANTA FE', 'DESARROLLO URBANO EL PIRU')
df_precios['Colonia'] = df_precios['Colonia'].str.replace('LOMAS DE BECERRA', 'LOMAS DE BECERRA GRANADA')
df_precios['Colonia'] = df_precios['Colonia'].str.replace('DANIEL GARZA', 'AMPL DANIEL GARZA')
df_precios['Colonia'] = df_precios['Colonia'].str.replace('SAN JUAN DE ARAGON I SECC', 'EJIDOS SAN JUAN DE ARAGON 1A SECC')
df_precios['Colonia'] = df_precios['Colonia'].str.replace('SAN JUAN DE ARAGON II SECC', 'SAN JUAN DE ARAGON EJIDOS 2A SECC')
df_precios['Colonia'] = df_precios['Colonia'].str.replace('SAN JUAN DE ARAGON III SECC', 'SAN JUAN DE ARAGON 3A SECC')
df_precios['Colonia'] = df_precios['Colonia'].str.replace('SAN JUAN DE ARAGON IV SECC', 'SAN JUAN DE ARAGON 4TA Y 5TA SECC')
df_precios['Colonia'] = df_precios['Colonia'].str.replace('SAN JUAN DE ARAGON VI SECC', 'SAN JUAN DE ARAGON 6TA SECC')
df_precios['Colonia'] = df_precios['Colonia'].str.replace('SAN JUAN DE ARAGON VII SECC', 'SAN JUAN DE ARAGON 7MA SECC')

df_precios.loc[
    (df_precios['Colonia'] == 'NIÑOS HEROES') & (df_precios['Municipio'] == 'BENITO JUÁREZ'),
    'Colonia'
] = 'NIÑOS HEROES DE CHAPULTEPEC'

In [ ]:
df_precios_checkpoint = df_precios.copy() # Guarda un punto de control del DataFrame de precios
df_map_checkpoint = df_map.copy() # Guarda un punto de control del DataFrame de servicios geográficos

In [ ]:
df_precios = df_precios_checkpoint.copy() # Restaura el DataFrame de precios desde el punto de control
df_map = df_map_checkpoint.copy() # Restaura el DataFrame de servicios geográficos desde el punto de control

In [ ]:
pip install unidecode rapidfuzz # Instala las librerías 'unidecode' para normalización de texto y 'rapidfuzz' para concordancia aproximada.

In [ ]:
import re
import pandas as pd
import numpy as np
from unidecode import unidecode # Para quitar acentos de las cadenas de texto
from rapidfuzz import process, fuzz # Para realizar concordancia de cadenas de texto aproximada (fuzzy matching)

# ---------- 1) Normalización general ----------
def clean_text(s):
    # Función para limpiar y estandarizar cadenas de texto
    if pd.isna(s):
        return s
    s = str(s).upper().strip() # Convertir a mayúsculas y quitar espacios al inicio/final
    s = unidecode(s)                # Quitar acentos (ej. 'Á' -> 'A')
    s = s.replace('.', '')          # Eliminar puntos
    s = s.replace(' - ', ' ')       # Reemplazar ' - ' por un espacio
    s = re.sub(r'\s+', ' ', s)      # Compactar múltiples espacios en uno solo
    return s

# Aplica la función de limpieza a las columnas relevantes de ambos dataframes
for df, cols in [
    (df_precios, ['Colonia', 'Municipio', 'Estado']),
    (df_map, ['SETT_NAME', 'MUN_NAME', 'ST_NAME'])
]:
    for c in cols:
        # Asegura que la columna sea de tipo string antes de aplicar la limpieza
        df[c] = df[c].astype(str).fillna('').map(clean_text)

# ---------- 2) Reemplazos especiales y armonización de Estados ----------
# Ajustes comunes que ya tenías y otros útiles
# Normalizar nombres de municipio (ejemplos)
# df_precios['Municipio'] = df_precios['Municipio'].replace({
#     'ALVARO OBREGON': 'ALVARO OBREGON'.upper(),  # si quieres mantener acentos con unidecode quedan sin acento
#     'BENITO JUAREZ': 'BENITO JUAREZ',
#     'COYOACAN': 'COYOACAN',
#     'CUAUHTEMOC': 'CUAUHTEMOC',
#     'TLAHUAC': 'TLAHUAC',
    # añade los reemplazos que ya usaste antes si los necesitas
# })

# Armonizar la columna 'Estado': mapear variantes a las etiquetas presentes en df_map
# Observación: df_map usa 'DISTRITO FEDERAL' y 'MEXICO' según la información previa
estado_map = {
    'CDMX': 'DISTRITO FEDERAL',
    'CIUDAD DE MEXICO': 'DISTRITO FEDERAL',
    'CIUDAD DE MEXICO': 'DISTRITO FEDERAL',
    'CIUDAD DE MÉXICO': 'DISTRITO FEDERAL',
    'CD. DE MEXICO': 'DISTRITO FEDERAL',
    'ESTADO DE MEXICO': 'MEXICO',
    'EDOMEX': 'MEXICO',
    'MEXICO': 'MEXICO',
    'DISTRITO FEDERAL': 'DISTRITO FEDERAL'
}
# Como ya aplicamos unidecode y uppercase, usar claves sin acento
estado_map = {k.upper(): v for k,v in estado_map.items()}
df_precios['Estado'] = df_precios['Estado'].replace(estado_map).fillna(df_precios['Estado'])

# Si df_map tiene otras etiquetas de estado (por ejemplo "CIUDAD DE MEXICO"), ajusta en consecuencia.
# También es conveniente inspeccionar valores únicos:
print("Estados únicos en df_map:", df_map['ST_NAME'].unique()[:20])
print("Estados únicos en df_precios (muestra):", pd.Series(df_precios['Estado'].unique()).tolist()[:20])

# ---------- 3) Reemplazos puntuales en nombres de colonia (ejemplos que ya usaste) ----------
# Aplica las reglas de estandarización de nombres (ej. '1RA' -> '1A', 'SECCION' -> 'SECC', etc.)
replacements = [
    ('SECCION','SECC'),
    ('1RA','1A'),
    ('2DA','2A'),
    ('3RA','3A'),
    ('EX-', 'EX '),
    ('BOSQUES','BOSQUE'),
    ('CENTRAL DE ABASTO','CENTRAL DE ABASTOS'),
    # añade las tuyas...
]

for old, new in replacements:
    df_map['SETT_NAME'] = df_map['SETT_NAME'].str.replace(old, new, regex=False)
    df_precios['Colonia'] = df_precios['Colonia'].str.replace(old, new, regex=False)

# Reglas puntuales que tenías (ejemplo)
df_precios.loc[
    (df_precios['Colonia'] == 'NINOS HEROES') & (df_precios['Municipio'] == 'BENITO JUAREZ'),
    'Colonia'
] = 'NINOS HEROES DE CHAPULTEPEC'.upper()


# ---------- 4) Merge principal ----------
# Define las columnas de df_map que se quieren añadir a df_precios
cols_to_add = ['SETT_NAME','MUN_NAME','ST_NAME','Hospitales','Escuelas','Esparcimiento','Restaurantes','Carpetas', 'Transporte', 'Parques', 'Latitud', 'Longitud']
map_subset = df_map[cols_to_add].copy() # Crea un subconjunto de df_map con estas columnas

# Realiza la fusión ('left merge') entre df_precios y el subconjunto de df_map
# Utiliza 'Colonia', 'Municipio', 'Estado' de df_precios y sus equivalentes de df_map como claves de fusión
df_merged = df_precios.merge(
    map_subset,
    how='left',
    left_on=['Colonia','Municipio','Estado'],
    right_on=['SETT_NAME','MUN_NAME','ST_NAME'],
    suffixes=('','_map') # Añade sufijos para diferenciar columnas con el mismo nombre si las hubiera
)

# Eliminar las columnas de match de df_map ya que se duplicarían o no son necesarias tras la fusión
df_merged = df_merged.drop(columns=['SETT_NAME','MUN_NAME','ST_NAME'])

# ---------- 5) Diagnóstico: cuántos coincidieron? ----------
print("Total filas df_precios:", len(df_precios))
# Muestra el número de valores no nulos para cada columna de servicio después de la fusión
for c in ['Hospitales','Escuelas','Esparcimiento','Restaurantes','Carpetas', 'Transporte', 'Parques', 'Latitud', 'Longitud']:
    print(c, "no nulos después de la fusión:", df_merged[c].notna().sum(), " / ", len(df_merged))

# Guarda un ejemplo de las filas que no encontraron una coincidencia para inspección manual
no_match = df_merged[df_merged['Hospitales'].isna()].copy()
no_match[['Colonia','Municipio','Estado']].drop_duplicates().head(50).to_csv('no_match_sample.csv', index=False)
print("Se guardó ejemplo de no-matches: no_match_sample.csv")

# ---------- 6) (Opcional) Fuzzy-match para colonias dentro del mismo Municipio+Estado ----------
# Esta función intenta encontrar la colonia más similar para las filas que no tuvieron una coincidencia exacta
def fill_with_fuzzy(df_left, df_right, left_on=('Colonia','Municipio','Estado'), right_on=('SETT_NAME','MUN_NAME','ST_NAME'), score_cutoff=92):
    # Construye un diccionario para agrupar SETT_NAME por (Municipio, Estado)
    groups = {}
    for (m, s), g in df_right.groupby(['MUN_NAME','ST_NAME']):
        groups[(m,s)] = g['SETT_NAME'].unique().tolist()

    filled = 0 # Contador de filas que se lograron rellenar con fuzzy matching
    # Itera sobre las filas de df_left que aún tienen valores nulos en la columna 'Hospitales'
    for idx, row in df_left[df_left['Hospitales'].isna()].iterrows():
        key = (row['Municipio'], row['Estado']) # Clave para buscar en el diccionario 'groups'
        candidates = groups.get(key) # Obtiene posibles colonias candidatas
        if not candidates: # Si no hay candidatas, se salta esta iteración
            continue
        # Realiza la concordancia aproximada para encontrar la mejor coincidencia
        best = process.extractOne(row['Colonia'], candidates, scorer=fuzz.WRatio)
        if best and best[1] >= score_cutoff: # Si se encuentra una buena coincidencia (score >= 92)
            chosen = best[0] # La colonia que mejor coincide
            # Obtiene los valores de df_map para la colonia elegida
            vals = df_right[(df_right['MUN_NAME']==key[0]) & (df_right['ST_NAME']==key[1]) & (df_right['SETT_NAME']==chosen)]
            if not vals.empty:
                # Toma la primera coincidencia (puede haber varias si hay duplicados)
                vals0 = vals.iloc[0]
                # Rellena las columnas de servicios en df_left
                df_left.at[idx, 'Hospitales'] = vals0['Hospitales']
                df_left.at[idx, 'Escuelas'] = vals0['Escuelas']
                df_left.at[idx, 'Esparcimiento'] = vals0['Esparcimiento']
                df_left.at[idx, 'Restaurantes'] = vals0['Restaurantes']
                df_left.at[idx, 'Carpetas'] = vals0['Carpetas']
                df_left.at[idx, 'Transporte'] = vals0['Transporte']
                df_left.at[idx, 'Parques'] = vals0['Parques']
                df_left.at[idx, 'Latitud'] = vals0['Latitud']
                df_left.at[idx, 'Longitud'] = vals0['Longitud']
                filled += 1 # Incrementa el contador de filas rellenadas
    return df_left, filled

df_filled, n_filled = fill_with_fuzzy(df_merged, df_map, score_cutoff=92) # Aplica el fuzzy matching
print("Fuzzy filled:", n_filled)
# Muestra el número de valores no nulos para cada columna de servicio después del fuzzy matching
for c in ['Hospitales','Escuelas','Esparcimiento','Restaurantes','Carpetas', 'Transporte', 'Parques', 'Latitud', 'Longitud']:
    print(c, "no nulos después del fuzzy matching:", df_filled[c].notna().sum())

# Finalmente guarda el resultado (línea comentada)
# df_filled.to_csv('df_precios_con_servicios.csv', index=False)
# print("Resultado guardado: df_precios_con_servicios.csv")

In [ ]:
df_filled.info() # Muestra un resumen de la información del DataFrame final, incluyendo tipos de datos y valores no nulos

In [ ]:
# Convierte la columna 'Fecha' a formato datetime para facilitar análisis temporales
df_filled['Fecha'] = pd.to_datetime(df_filled['Fecha'])

In [ ]:
df_filled.info() # Vuelve a mostrar la información del DataFrame para verificar el cambio de tipo de 'Fecha'

In [22]:
# Ajusta las opciones de visualización de Pandas para mostrar más filas y columnas completas al imprimir DataFrames
pd.set_option('display.max_rows', 10) # Mostrar un máximo de 10 filas
pd.set_option('display.max_columns', 10) # Mostrar un máximo de 10 columnas

In [ ]:
df_filled[df_filled['Municipio'] == 'ALVARO OBREGON'] # Filtra el DataFrame para mostrar solo las propiedades del municipio 'ALVARO OBREGON'

In [ ]:
df_filled.to_csv('df_precios_con_servicios.csv', index=False) # Guarda el DataFrame final enriquecido en un archivo CSV